# Choosing K with K-Means, Elbow and Silhouette Analysis

Companion to the [complete article](https://todatabeyond.com/blog/stop-using-elbow-diagram-to-find-best-k-value-and-use-this-instead). Run all cells in order with Python 3.11+ in Jupyter/Colab. Setup installs pinned NumPy 2.3.3, SciPy 1.16.2, scikit-learn 1.7.2, Matplotlib 3.10.9, joblib 1.5.2 and threadpoolctl 3.6.0 into the current interpreter. CPU only, no API keys, GPU or datasets to download. Only package installation needs network access. Full original gist snapshots and retrieved revisions are in the nonexecuting appendix.

Reproduce exactly the original five centers, standard deviations, 2000 points and make_blobs random_state=7. Generating labels are printed only to check data creation; they are not passed into unsupervised fitting or used as a metric selector. Make all KMeans calls explicit with n_init=50 and algorithm=lloyd rather than inheriting changed modern defaults. Use fifty restarts uniformly to reduce sensitivity to local minima; this is not a global-optimum guarantee. Keep random_state=42 and default iteration/tolerance settings. Cluster numbers are arbitrary labels; source screenshot numbering or color permutations are not correctness errors.

Supply missing np/plt/mpl imports, an isolated temporary save_fig helper and explicit print/show output. Keep the original plotting functions, one-million-point decision grids and all seven figure workflows. Named voronoi_plot.png is saved only in an agent-created temporary directory; inline plots are displayed. The elbow/score curves now show all tested k values and use visible data ranges instead of clipping k=9 or the highest inertia. Source screenshots are retained unchanged in the blog, alongside fresh figures.

Original public prose is preserved. Private findings: sklearn inertia is a SUM of squared nearest-centroid distances, not the source's mean. Silhouette requires 2 through n-1 observed clusters and assigns singleton points zero. On this data the mean-score winner is k=4, not the five generating blobs; silhouette does not universally recover a true K. Per-cluster sizes, separation, feature scaling, domain purpose and K-Means assumptions matter. No universal superiority, ground-truth recovery or speed benchmark is claimed. The source's Figure 7 discussion fixes a numeric label/vertical order that may not survive arbitrary label permutations; use geometry and sizes, not that label as a stable semantic identity.

Regression compares every fitted inertia and label against independent squared distances and all 16000 coefficients for k=2..9 against a full NumPy Euclidean-distance implementation. It also checks deterministic data/refits, singleton/invalid cluster counts, label/order invariance and boundary predictions. The default experiment uses full data, not a sampled silhouette approximation. Committed notebook outputs remain empty/counts null.


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--disable-pip-version-check', 'numpy==2.3.3', 'scipy==1.16.2',
                'scikit-learn==1.7.2', 'matplotlib==3.10.9',
                'joblib==1.5.2', 'threadpoolctl==3.6.0'], check=True)

import hashlib
import tempfile
from pathlib import Path
import numpy as np
import scipy
import sklearn
import matplotlib as mpl
import matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits

%matplotlib inline
thread_limits = threadpool_limits(limits=1)
np.set_printoptions(precision=8, threshold=1000, linewidth=120)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
figure_directory = Path(tempfile.mkdtemp(prefix='tdb-kmeans-'))
def save_fig(name):
    path = figure_directory / (name + '.png')
    plt.gcf().savefig(path, dpi=150, bbox_inches='tight')
    return path

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__, '| SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__, '| Matplotlib:', mpl.__version__)
print('CPU, one numerical thread; synthetic data only, no API keys or external dataset.')


## 1. Generate Original Blobs


In [ ]:
from sklearn.datasets import make_blobs
blob_centers = np.array(
    [[ 0.2,  2.3],
     [-1.5 ,  2.3],
     [-2.8,  1.8],
     [-2.8,  2.8],
     [-2.8,  1.3]])
blob_std = np.array([0.4, 0.3, 0.1, 0.1, 0.1])
X, y = make_blobs(n_samples=2000, centers=blob_centers,
                  cluster_std=blob_std, random_state=7)
print("Dataset shape:", X.shape)
print("Generating group sizes:", np.bincount(y).tolist())


## 2. Plot Unlabeled Data


In [ ]:
def plot_clusters(X, y=None):
    plt.scatter(X[:, 0], X[:, 1], c=y, s=1)
    plt.xlabel("$x_1$", fontsize=14)
    plt.ylabel("$x_2$", fontsize=14, rotation=0)
    
plt.figure(figsize=(10, 8))
plot_clusters(X)
plt.show()
print("Unlabeled scatter: all", len(X), "points.")


## 3. Fit Five Clusters


In [ ]:
from sklearn.cluster import KMeans
k = 5
kmeans = KMeans(n_clusters=k, random_state=42, n_init=50, algorithm='lloyd')
y_pred = kmeans.fit_predict(X)
print("Fitted k:", kmeans.n_clusters, "| Cluster sizes:", np.bincount(y_pred).tolist())


## 4. Inspect Cluster Labels


In [ ]:
print(repr(y_pred))


## 5. Inspect Cluster Centers


In [ ]:
print(repr(kmeans.cluster_centers_))


## 6. Define Plot Helpers


In [ ]:
def plot_data(X):
    plt.plot(X[:, 0], X[:, 1], 'k.', markersize=2)

def plot_centroids(centroids, weights=None, circle_color='w', cross_color='k'):
    if weights is not None:
        centroids = centroids[weights > weights.max() / 10]
    plt.scatter(centroids[:, 0], centroids[:, 1],
                marker='o', s=35, linewidths=8,
                color=circle_color, zorder=10, alpha=0.9)
    plt.scatter(centroids[:, 0], centroids[:, 1],
                marker='x', s=2, linewidths=12,
                color=cross_color, zorder=11, alpha=1)


def plot_decision_boundaries(clusterer, X, resolution=1000, show_centroids=True,
                             show_xlabels=True, show_ylabels=True):
    mins = X.min(axis=0) - 0.1
    maxs = X.max(axis=0) + 0.1
    xx, yy = np.meshgrid(np.linspace(mins[0], maxs[0], resolution),
                         np.linspace(mins[1], maxs[1], resolution))
    Z = clusterer.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.contourf(Z, extent=(mins[0], maxs[0], mins[1], maxs[1]),
                cmap="Pastel2")
    plt.contour(Z, extent=(mins[0], maxs[0], mins[1], maxs[1]),
                linewidths=1, colors='k')
    plot_data(X)
    if show_centroids:
        plot_centroids(clusterer.cluster_centers_)

    if show_xlabels:
        plt.xlabel("$x_1$", fontsize=14)
    else:
        plt.tick_params(labelbottom=False)
    if show_ylabels:
        plt.ylabel("$x_2$", fontsize=14, rotation=0)
    else:
        plt.tick_params(labelleft=False)


def plot_clusterer_comparison(clusterer1, clusterer2, X, title1=None, title2=None):
    clusterer1.fit(X)
    clusterer2.fit(X)

    plt.figure(figsize=(10, 3.2))

    plt.subplot(121)
    plot_decision_boundaries(clusterer1, X)
    if title1:
        plt.title(title1, fontsize=14)

    plt.subplot(122)
    plot_decision_boundaries(clusterer2, X, show_ylabels=False)
    if title2:
        plt.title(title2, fontsize=14)
print("Plot helpers ready; default decision grid is 1000 by 1000.")


## 7. Plot Five-Cluster Boundaries


In [ ]:
plt.figure(figsize=(8, 4))
plot_decision_boundaries(kmeans, X)
save_fig("voronoi_plot")
plt.show()
print("Five-centroid decision boundaries plotted; voronoi_plot.png saved in the temporary figure directory.")


## 8. Compare Three and Eight Clusters


In [ ]:
from sklearn.cluster import KMeans

kmeans_k3 = KMeans(n_clusters=3, random_state=42, n_init=50, algorithm='lloyd')
kmeans_k8 = KMeans(n_clusters=8, random_state=42, n_init=50, algorithm='lloyd')

plot_clusterer_comparison(kmeans_k3, kmeans_k8, X, "$k=3$", "$k=8$")
plt.show()
print("Comparison: k=3 versus k=8, with all 2000 data points in each panel.")


## 9. Plot Inertia and Elbow


In [ ]:
kmeans_per_k = [KMeans(n_clusters=k, random_state=42, n_init=50, algorithm='lloyd').fit(X)
                for k in range(1, 10)]

inertias = [model.inertia_ for model in kmeans_per_k]


plt.figure(figsize=(10, 5))
plt.plot(range(1, 10), inertias, "bo-")
plt.xlabel("$k$", fontsize=14)
plt.ylabel("Inertia", fontsize=14)
plt.annotate('Elbow',
             xy=(4, inertias[3]),
             xytext=(0.55, 0.55),
             textcoords='figure fraction',
             fontsize=16,
             arrowprops=dict(facecolor='black', shrink=0.1)
            )
plt.axis([0.8, 9.2, 0, max(inertias) * 1.05])
plt.title('The Elbow Diagram')
plt.show()
for count, inertia in enumerate(inertias, start=1):
    print(f"k={count}: inertia={inertia:.6f}")


## 10. Plot Four-Cluster Boundaries


In [ ]:
plt.figure(figsize=(8, 4))
plot_decision_boundaries(kmeans_per_k[4-1], X)
plt.show()
print("Four-centroid decision boundaries plotted.")


## 11. Plot Mean Silhouette Scores


In [ ]:
from sklearn.metrics import silhouette_score
silhouette_scores = [silhouette_score(X, model.labels_)
                     for model in kmeans_per_k[1:]]
                     
plt.figure(figsize=(8,8))
plt.plot(range(2, 10), silhouette_scores, "bo-")
plt.xlabel("$k$", fontsize=14)
plt.ylabel("Silhouette score", fontsize=14)
plt.axis([1.8, 9.2, min(silhouette_scores) - 0.02, max(silhouette_scores) + 0.02])
plt.title('The Silhouette score')
plt.show()
for count, score in enumerate(silhouette_scores, start=2):
    print(f"k={count}: silhouette={score:.8f}")
print("Highest mean silhouette among tested k:", int(np.argmax(silhouette_scores)) + 2)


## 12. Plot Per-Cluster Silhouettes


In [ ]:
from sklearn.metrics import silhouette_samples
from matplotlib.ticker import FixedLocator, FixedFormatter

plt.figure(figsize=(11, 9))

for k in (3, 4, 5, 6):
    plt.subplot(2, 2, k - 2)
    
    cluster_labels = kmeans_per_k[k - 1].labels_
    silhouette_coefficients = silhouette_samples(X, cluster_labels)

    padding = len(X) // 30
    pos = padding
    ticks = []
    for i in range(k):
        coeffs = silhouette_coefficients[cluster_labels == i]
        coeffs.sort()

        color = plt.get_cmap("Spectral")(i / k)
        plt.fill_betweenx(np.arange(pos, pos + len(coeffs)), 0, coeffs,
                          facecolor=color, edgecolor=color, alpha=0.7)
        ticks.append(pos + len(coeffs) // 2)
        pos += len(coeffs) + padding

    plt.gca().yaxis.set_major_locator(FixedLocator(ticks))
    plt.gca().yaxis.set_major_formatter(FixedFormatter([str(i) for i in range(k)]))
    if k in (3, 5):
        plt.ylabel("Cluster")
    
    if k in (5, 6):
        plt.gca().set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])
        plt.xlabel("Silhouette Coefficient")
    else:
        plt.tick_params(labelbottom=False)

    plt.axvline(x=silhouette_scores[k - 2], color="red", linestyle="--")
    plt.title("$k={}$".format(k), fontsize=16)

plt.show()
for count in (3, 4, 5, 6):
    model = kmeans_per_k[count - 1]
    print(f"k={count}: cluster sizes={np.bincount(model.labels_).tolist()}, mean silhouette={silhouette_scores[count - 2]:.8f}")


## Independent full-data regressions


In [ ]:
from sklearn.metrics import silhouette_score, silhouette_samples

X_again, y_again = make_blobs(n_samples=2000, centers=blob_centers,
                             cluster_std=blob_std, random_state=7)
assert X.shape == (2000, 2) and np.isfinite(X).all()
assert np.array_equal(X, X_again) and np.array_equal(y, y_again)
assert np.bincount(y).tolist() == [400] * 5
assert np.array_equal(y_pred, kmeans.labels_) and np.array_equal(y_pred, kmeans.predict(X))
assert kmeans.n_init == 50 and kmeans.algorithm == 'lloyd'
repeat = KMeans(n_clusters=5, random_state=42, n_init=50, algorithm='lloyd').fit(X)
assert np.array_equal(repeat.labels_, kmeans.labels_)
np.testing.assert_allclose(repeat.cluster_centers_, kmeans.cluster_centers_, atol=1e-12)

# Archived centroid screenshot, compared geometrically up to label permutation.
archived_centers = np.array([[0.20876306, 2.25551336], [-2.80389616, 1.80117999],
                             [-1.46679593, 2.28585348], [-2.79290307, 2.79641063],
                             [-2.80037642, 1.30082566]])
np.testing.assert_allclose(sorted(kmeans.cluster_centers_.tolist()),
                           sorted(archived_centers.tolist()), atol=1e-8, rtol=0)

for count, model in enumerate(kmeans_per_k, start=1):
    assert model.n_clusters == count and model.n_init == 50
    assert set(model.labels_) == set(range(count)) and len(model.labels_) == len(X)
    squared = ((X[:, None, :] - model.cluster_centers_[None, :, :]) ** 2).sum(axis=2)
    assert np.array_equal(np.argmin(squared, axis=1), model.labels_)
    independent_inertia = squared[np.arange(len(X)), model.labels_].sum()
    np.testing.assert_allclose(model.inertia_, independent_inertia, rtol=1e-12, atol=1e-10)
assert np.all(np.diff(inertias) < 0)

# Full independent Euclidean distances, not sklearn's pairwise/silhouette helpers.
delta = X[:, None, :] - X[None, :, :]
distances = np.sqrt(np.sum(delta ** 2, axis=2))
del delta
np.fill_diagonal(distances, 0)
def manual_coefficients(distances, labels):
    labels = np.asarray(labels)
    groups = np.unique(labels)
    if not 2 <= len(groups) < len(labels):
        raise ValueError('Silhouette requires 2 through n-1 clusters.')
    output = np.zeros(len(labels))
    for group in groups:
        selected = np.flatnonzero(labels == group)
        if len(selected) == 1:
            continue
        within = distances[np.ix_(selected, selected)].sum(axis=1) / (len(selected) - 1)
        nearest = np.min(np.column_stack([
            distances[np.ix_(selected, np.flatnonzero(labels == other))].mean(axis=1)
            for other in groups if other != group]), axis=1)
        denominator = np.maximum(within, nearest)
        output[selected] = np.divide(nearest - within, denominator,
                                     out=np.zeros_like(within), where=denominator != 0)
    return output

for count, model in enumerate(kmeans_per_k[1:], start=2):
    manual = manual_coefficients(distances, model.labels_)
    actual = silhouette_samples(X, model.labels_)
    np.testing.assert_allclose(actual, manual, atol=1e-10, rtol=1e-10)
    np.testing.assert_allclose(silhouette_scores[count - 2], manual.mean(), atol=1e-10)
    np.testing.assert_allclose(silhouette_score(X, 17 * model.labels_ + 3), manual.mean(), atol=1e-10)
assert int(np.argmax(silhouette_scores)) + 2 == 4
assert silhouette_scores[2] > silhouette_scores[3]
assert kmeans_per_k[4].n_clusters == 5  # Generating blob count is not the metric winner.

probe = np.random.default_rng(42).uniform(X.min(axis=0), X.max(axis=0), size=(128, 2))
for model in [kmeans, kmeans_k3, kmeans_k8, kmeans_per_k[3]]:
    expected = ((probe[:, None, :] - model.cluster_centers_[None, :, :]) ** 2).sum(axis=2).argmin(axis=1)
    assert np.array_equal(model.predict(probe), expected)

small = np.array([[0., 0.], [0., 1.], [10., 0.], [10., 1.], [30., 0.]])
small_labels = np.array([0, 0, 1, 1, 2])
small_distances = np.sqrt(((small[:, None, :] - small[None, :, :]) ** 2).sum(axis=2))
small_manual = manual_coefficients(small_distances, small_labels)
np.testing.assert_allclose(silhouette_samples(small, small_labels), small_manual, atol=1e-12)
assert small_manual[-1] == 0  # Singleton convention.
permutation = np.array([4, 2, 0, 3, 1])
np.testing.assert_allclose(silhouette_samples(small[permutation], small_labels[permutation]), small_manual[permutation], atol=1e-12)
for labels in [np.zeros(len(small)), np.arange(len(small))]:
    try:
        silhouette_score(small, labels)
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid cluster count unexpectedly accepted.')
saved = figure_directory / 'voronoi_plot.png'
assert saved.exists() and saved.read_bytes().startswith(b'\x89PNG\r\n\x1a\n')
data_sha = hashlib.sha256(np.asarray(X, dtype='<f8').tobytes()).hexdigest()
print('All twelve original gist examples passed on all 2000 source-generated points; seven plots rendered.')
print('All nine inertias and nearest-centroid labels match independent squared-distance calculations.')
print('All 16000 silhouette coefficients for k=2..9 and eight means match an independent full-distance oracle.')
print('Highest mean silhouette: k=4; five generating blobs remain a separate modeling choice.')
print('Deterministic refits, relabeling/order invariance, singleton/invalid-label edges and 128 boundary probes pass.')
print('Dataset SHA-256:', data_sha)
thread_limits.restore_original_limits()


## Complete original gist snapshots (not executed)

### Generate Original Blobs

[5794d1fce0c5209440445ce51aaf4a21](https://gist.github.com/youssefHosni/5794d1fce0c5209440445ce51aaf4a21) · revision `cef1bbcb576d2f4bab466bfdffb65117f57f9975`

```python
from sklearn.datasets import make_blobs
blob_centers = np.array(
    [[ 0.2,  2.3],
     [-1.5 ,  2.3],
     [-2.8,  1.8],
     [-2.8,  2.8],
     [-2.8,  1.3]])
blob_std = np.array([0.4, 0.3, 0.1, 0.1, 0.1])
X, y = make_blobs(n_samples=2000, centers=blob_centers,
                  cluster_std=blob_std, random_state=7)
```

### Plot Unlabeled Data

[b9d1a662108ac207e24394b3f02bf668](https://gist.github.com/youssefHosni/b9d1a662108ac207e24394b3f02bf668) · revision `cd0ff43213cd1e6c8c93ba3ef512a58c4ecc7a61`

```python
def plot_clusters(X, y=None):
    plt.scatter(X[:, 0], X[:, 1], c=y, s=1)
    plt.xlabel("$x_1$", fontsize=14)
    plt.ylabel("$x_2$", fontsize=14, rotation=0)
    
plt.figure(figsize=(10, 8))
plot_clusters(X)
plt.show()    
```

### Fit Five Clusters

[cc407f0f45b946d0e6ba1e08b48dc4d8](https://gist.github.com/youssefHosni/cc407f0f45b946d0e6ba1e08b48dc4d8) · revision `ac55a8ca1af3a4a6874069dae3f77a3696f4699a`

```python
from sklearn.cluster import KMeans
k = 5
kmeans = KMeans(n_clusters=k, random_state=42)
y_pred = kmeans.fit_predict(X)
```

### Inspect Cluster Labels

[adfecabf2e08bf4008c6c74d52a40bff](https://gist.github.com/youssefHosni/adfecabf2e08bf4008c6c74d52a40bff) · revision `78826ef0f1987558d925a495669db637a8058a55`

```python
y_pred
```

### Inspect Cluster Centers

[4b9f470b34ad97eaa620eb17e4fdf0c1](https://gist.github.com/youssefHosni/4b9f470b34ad97eaa620eb17e4fdf0c1) · revision `f5f9abcbe5b90374250e790f4cc2d546c0dac9ee`

```python
kmeans.cluster_centers_
```

### Define Plot Helpers

[f2fa8046a023f662698096c74f995cc5](https://gist.github.com/youssefHosni/f2fa8046a023f662698096c74f995cc5) · revision `dc415adb53f0529615c03eb3cfe912a50ee9027b`

```python
def plot_data(X):
    plt.plot(X[:, 0], X[:, 1], 'k.', markersize=2)

def plot_centroids(centroids, weights=None, circle_color='w', cross_color='k'):
    if weights is not None:
        centroids = centroids[weights > weights.max() / 10]
    plt.scatter(centroids[:, 0], centroids[:, 1],
                marker='o', s=35, linewidths=8,
                color=circle_color, zorder=10, alpha=0.9)
    plt.scatter(centroids[:, 0], centroids[:, 1],
                marker='x', s=2, linewidths=12,
                color=cross_color, zorder=11, alpha=1)


def plot_decision_boundaries(clusterer, X, resolution=1000, show_centroids=True,
                             show_xlabels=True, show_ylabels=True):
    mins = X.min(axis=0) - 0.1
    maxs = X.max(axis=0) + 0.1
    xx, yy = np.meshgrid(np.linspace(mins[0], maxs[0], resolution),
                         np.linspace(mins[1], maxs[1], resolution))
    Z = clusterer.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.contourf(Z, extent=(mins[0], maxs[0], mins[1], maxs[1]),
                cmap="Pastel2")
    plt.contour(Z, extent=(mins[0], maxs[0], mins[1], maxs[1]),
                linewidths=1, colors='k')
    plot_data(X)
    if show_centroids:
        plot_centroids(clusterer.cluster_centers_)

    if show_xlabels:
        plt.xlabel("$x_1$", fontsize=14)
    else:
        plt.tick_params(labelbottom=False)
    if show_ylabels:
        plt.ylabel("$x_2$", fontsize=14, rotation=0)
    else:
        plt.tick_params(labelleft=False)


def plot_clusterer_comparison(clusterer1, clusterer2, X, title1=None, title2=None):
    clusterer1.fit(X)
    clusterer2.fit(X)

    plt.figure(figsize=(10, 3.2))

    plt.subplot(121)
    plot_decision_boundaries(clusterer1, X)
    if title1:
        plt.title(title1, fontsize=14)

    plt.subplot(122)
    plot_decision_boundaries(clusterer2, X, show_ylabels=False)
    if title2:
        plt.title(title2, fontsize=14)
```

### Plot Five-Cluster Boundaries

[9651f77745a15985e3fcdbd39b04621d](https://gist.github.com/youssefHosni/9651f77745a15985e3fcdbd39b04621d) · revision `7f65605a000f1076ca87633c1b2f15ab93d955e7`

```python
plt.figure(figsize=(8, 4))
plot_decision_boundaries(kmeans, X)
save_fig("voronoi_plot")
plt.show()
```

### Compare Three and Eight Clusters

[16131eb3b69ad66deee10f854ccc67a5](https://gist.github.com/youssefHosni/16131eb3b69ad66deee10f854ccc67a5) · revision `b21ebdfed25f6fe2fa7e192817ddb04c3e934960`

```python
from sklearn.cluster import KMeans

kmeans_k3 = KMeans(n_clusters=3, random_state=42)
kmeans_k8 = KMeans(n_clusters=8, random_state=42)

plot_clusterer_comparison(kmeans_k3, kmeans_k8, X, "$k=3$", "$k=8$")
plt.show()
```

### Plot Inertia and Elbow

[67cd2ea63493de08d8a71b99cd6e1511](https://gist.github.com/youssefHosni/67cd2ea63493de08d8a71b99cd6e1511) · revision `e1672a21522935fa3aa0485cf5a2f1e7bca35329`

```python
kmeans_per_k = [KMeans(n_clusters=k, random_state=42).fit(X)
                for k in range(1, 10)]

inertias = [model.inertia_ for model in kmeans_per_k]


plt.figure(figsize=(10, 5))
plt.plot(range(1, 10), inertias, "bo-")
plt.xlabel("$k$", fontsize=14)
plt.ylabel("Inertia", fontsize=14)
plt.annotate('Elbow',
             xy=(4, inertias[3]),
             xytext=(0.55, 0.55),
             textcoords='figure fraction',
             fontsize=16,
             arrowprops=dict(facecolor='black', shrink=0.1)
            )
plt.axis([1, 8.5, 0, 1300])
plt.title('The Elbow Diagram')
plt.show()
```

### Plot Four-Cluster Boundaries

[072dd26bed8ab21af957ae42e350fc40](https://gist.github.com/youssefHosni/072dd26bed8ab21af957ae42e350fc40) · revision `d62a2bcc5bd4bbc4dc13755e926945e99ce9cd4d`

```python
plot_decision_boundaries(kmeans_per_k[4-1], X)
plt.show()
```

### Plot Mean Silhouette Scores

[58a9c78b9d7f687a8ce937367a9b233a](https://gist.github.com/youssefHosni/58a9c78b9d7f687a8ce937367a9b233a) · revision `f9e417ac0c366c01868c01b78935acef0d13fa43`

```python
from sklearn.metrics import silhouette_score
silhouette_scores = [silhouette_score(X, model.labels_)
                     for model in kmeans_per_k[1:]]
                     
plt.figure(figsize=(8,8))
plt.plot(range(2, 10), silhouette_scores, "bo-")
plt.xlabel("$k$", fontsize=14)
plt.ylabel("Silhouette score", fontsize=14)
plt.axis([1.8, 8.5, 0.55, 0.7])
plt.title('The Silhouette score')
plt.show()                     
```

### Plot Per-Cluster Silhouettes

[e0bb9bbe13a36e1f528a4f9fd4480983](https://gist.github.com/youssefHosni/e0bb9bbe13a36e1f528a4f9fd4480983) · revision `fd6cfbc681435305e99a0cb6b2692330f89c1861`

```python
from sklearn.metrics import silhouette_samples
from matplotlib.ticker import FixedLocator, FixedFormatter

plt.figure(figsize=(11, 9))

for k in (3, 4, 5, 6):
    plt.subplot(2, 2, k - 2)
    
    y_pred = kmeans_per_k[k - 1].labels_
    silhouette_coefficients = silhouette_samples(X, y_pred)

    padding = len(X) // 30
    pos = padding
    ticks = []
    for i in range(k):
        coeffs = silhouette_coefficients[y_pred == i]
        coeffs.sort()

        color = mpl.cm.Spectral(i / k)
        plt.fill_betweenx(np.arange(pos, pos + len(coeffs)), 0, coeffs,
                          facecolor=color, edgecolor=color, alpha=0.7)
        ticks.append(pos + len(coeffs) // 2)
        pos += len(coeffs) + padding

    plt.gca().yaxis.set_major_locator(FixedLocator(ticks))
    plt.gca().yaxis.set_major_formatter(FixedFormatter(range(k)))
    if k in (3, 5):
        plt.ylabel("Cluster")
    
    if k in (5, 6):
        plt.gca().set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])
        plt.xlabel("Silhouette Coefficient")
    else:
        plt.tick_params(labelbottom=False)

    plt.axvline(x=silhouette_scores[k - 2], color="red", linestyle="--")
    plt.title("$k={}$".format(k), fontsize=16)

plt.show()
```
